# Part 1: Charlotte Restaurant Rating Prediction with SageMaker

## Scenario
You are a data scientist at a Charlotte-based food tech startup. Your team has collected data on restaurants across Charlotte neighborhoods. Leadership wants a deployed ML model that predicts restaurant ratings from structured features — so they can estimate ratings for new restaurants before reviews come in.

## What You'll Do
1. **Explore** the Charlotte restaurant dataset
2. **Prepare** data for SageMaker (feature engineering, S3 upload)
3. **Train** an XGBoost model using SageMaker's built-in algorithm
4. **Tune** hyperparameters using SageMaker's automatic tuning
5. **Deploy** the best model to a real-time endpoint
6. **Test** predictions on new data

## Important
- **Do NOT change instance types** — they are set to work within Learner Lab limits
- Save your work frequently — the lab session may time out
- You will export this notebook (with outputs) as a submission artifact


## Setup

In [ ]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import os
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

# SageMaker session and role
role = get_execution_role()
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'clt-restaurant-ml'
region = boto3.Session().region_name

print(f"Role: {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")


## Load the Dataset

In [ ]:
# Download the dataset files from the course S3 bucket
!aws s3 cp s3://dsba6190-smith-lab3/data/charlotte_restaurants.csv ./
!aws s3 cp s3://dsba6190-smith-lab3/data/charlotte_reviews.csv ./

df = pd.read_csv('charlotte_restaurants.csv')
print(f"Loaded {len(df)} restaurants")
print(f"Columns: {list(df.columns)}")
df.head()


In [ ]:
# Quick overview of the data
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
df.describe()


## Phase 1A: Data Exploration & Preparation

### Explore the Data (~20 min)

Create **at least 2 visualizations** that explore the relationship between features and `avg_rating` (the target variable).

**Suggestions** (pick 2-3, or create your own):
- Rating distribution by neighborhood (boxplot)
- Cuisine type vs. average rating (bar chart)
- Price range vs. rating (violin plot or scatter)
- Correlation heatmap of numerical features
- Review count vs. rating (scatter plot)


In [ ]:
# TODO: Create your visualizations here
# =============================================
# Visualization 1:
# =============================================


# =============================================
# Visualization 2:
# =============================================


# =============================================
# Visualization 3 (optional):
# =============================================



### Feature Engineering

Prepare the data for XGBoost training. You need to:
1. Select which columns to use as features
2. One-hot encode categorical columns (cuisine, neighborhood)
3. Separate the target variable (`avg_rating`)
4. Split into train/test sets


In [ ]:
# TODO: Select your feature columns
# Think about which columns are useful predictors vs. identifiers
# =============================================
feature_cols = []  # TODO: Fill in the columns you want to use as features
# Example: ['price_range', 'review_count', 'has_outdoor_seating', ...]
# Remember: 'restaurant_id', 'name', 'lat', 'lon' are identifiers, not features
# =============================================

target_col = 'avg_rating'

# One-hot encode categorical columns
categorical_cols = [c for c in feature_cols if df[c].dtype == 'object']
numerical_cols = [c for c in feature_cols if df[c].dtype != 'object']

print(f"Categorical features to encode: {categorical_cols}")
print(f"Numerical features: {numerical_cols}")

# Create feature matrix
df_encoded = pd.get_dummies(df[feature_cols], columns=categorical_cols, drop_first=True)
print(f"\nFeature matrix shape after encoding: {df_encoded.shape}")
df_encoded.head()


In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 80% train, 20% test
X = df_encoded.values.astype('float32')
y = df[target_col].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples")

# SageMaker XGBoost expects: target column FIRST, then features, no header
train_data = np.column_stack([y_train, X_train])
test_data = np.column_stack([y_test, X_test])

# Save to CSV
np.savetxt('train.csv', train_data, delimiter=',', fmt='%.6f')
np.savetxt('test.csv', test_data, delimiter=',', fmt='%.6f')

# Upload to S3
train_s3 = session.upload_data('train.csv', bucket=bucket, key_prefix=f'{prefix}/data/train')
test_s3 = session.upload_data('test.csv', bucket=bucket, key_prefix=f'{prefix}/data/test')

print(f"\nUploaded training data to: {train_s3}")
print(f"Uploaded test data to: {test_s3}")
print("\n📸 SCREENSHOT: Open S3 in the AWS Console and take a screenshot showing these files.")


## Phase 1B: Train XGBoost on SageMaker

Now we'll train an XGBoost model using SageMaker's **built-in algorithm**. This is different from training locally with scikit-learn:

- SageMaker spins up a **separate ML instance** to run training
- Data is read from **S3** (not local memory)
- The trained model artifact is saved back to **S3** as a `.tar.gz` file
- You pay only for the training time (billed by the second)


In [ ]:
# Configure the XGBoost estimator
# DO NOT change the instance_type — ml.m5.large is the correct size for Learner Lab
xgb_image = sagemaker.image_uris.retrieve('xgboost', region, version='1.7-1')

xgb = sagemaker.estimator.Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',  # DO NOT CHANGE — Learner Lab restriction
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session
)

# TODO: Set your initial hyperparameters
# =============================================
# Consider:
#   objective: 'reg:squarederror' (we're predicting a continuous rating)
#   num_round: number of boosting rounds (try 100-300)
#   max_depth: tree depth (try 3-8)
#   eta: learning rate (try 0.05-0.3)
#   eval_metric: 'rmse' (root mean squared error)
xgb.set_hyperparameters(
    # TODO: Fill in your hyperparameters
)
# =============================================


In [ ]:
# Define the data channels
train_input = TrainingInput(train_s3, content_type='text/csv')
test_input = TrainingInput(test_s3, content_type='text/csv')

# Launch the training job — this takes about 5 minutes
print("Starting training job... (this will take ~5 minutes)")
xgb.fit({'train': train_input, 'validation': test_input})
print("Training complete!")


### How SageMaker Training Differs from Local Training

| Aspect | Local (scikit-learn) | SageMaker |
|--------|---------------------|-----------|
| **Compute** | Your laptop's CPU/RAM | Dedicated ML instance (ml.m5.large = 2 vCPUs, 8 GB RAM) |
| **Data** | Loaded into memory from disk | Streamed from S3 |
| **Model Storage** | Python pickle file on disk | Compressed artifact (.tar.gz) on S3 |
| **Scaling** | Limited by your hardware | Change `instance_type` or `instance_count` |
| **Cost** | Free (your machine) | Pay per second of training time |
| **Reproducibility** | Depends on your environment | Containerized, fully reproducible |

**Why does this matter for data analytics?** When your datasets grow beyond what fits in a laptop's memory (common in real-world analytics), SageMaker lets you scale training to instances with hundreds of GBs of RAM or GPU clusters — without changing your code. You just change the `instance_type`.


## Phase 1C: Hyperparameter Tuning

Instead of manually trying different hyperparameter values (grid search), SageMaker can **automatically search** for the best combination using Bayesian optimization. This is especially powerful in the cloud because it can run multiple training jobs **in parallel**.


In [ ]:
# TODO: Define your hyperparameter ranges
# =============================================
# Think about what ranges make sense for each parameter.
# The tuner will search within these ranges.
#
# Hints:
#   max_depth: controls tree complexity (too deep = overfitting)
#   eta: learning rate (smaller = more conservative, needs more rounds)
#   min_child_weight: minimum data in a leaf (higher = more conservative)
#   subsample: fraction of data per tree (lower = more regularization)

hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 10),       # TODO: adjust range if desired
    'eta': ContinuousParameter(0.05, 0.3),      # TODO: adjust range if desired
    'min_child_weight': IntegerParameter(1, 10), # TODO: adjust range if desired
    'subsample': ContinuousParameter(0.5, 1.0),  # TODO: adjust range if desired
}
# =============================================

# Configure the tuner
# DO NOT change max_jobs or max_parallel_jobs — set for Learner Lab cost management
tuner = HyperparameterTuner(
    estimator=xgb,
    objective_metric_name='validation:rmse',
    objective_type='Minimize',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=6,                # DO NOT CHANGE — controls cost
    max_parallel_jobs=2,       # DO NOT CHANGE — controls cost
    strategy='Bayesian'
)

print("Starting hyperparameter tuning... (this will take ~8 minutes)")
tuner.fit({'train': train_input, 'validation': test_input})
print("Tuning complete!")


### While you wait: Reflection Questions

Think about these questions (you don't need to write answers, but they'll help with the analysis later):

1. **Why is cloud-based tuning advantageous over local grid search?** Consider: parallel execution, instance flexibility, and automatic experiment tracking.
2. **Bayesian optimization vs. grid search:** How does the tuner decide which combinations to try next? (Hint: it uses results from earlier jobs to guide later ones.)
3. **Cost tradeoff:** We're running 6 jobs with 2 parallel. What would happen to cost and time if we ran 20 jobs with 5 parallel? When would that be worth it?


In [ ]:
# Analyze tuning results
tuner_results = tuner.analytics().dataframe()
print(f"Tuning completed: {len(tuner_results)} jobs")
print(f"\nBest RMSE: {tuner_results['FinalObjectiveValue'].min():.4f}")
print(f"Worst RMSE: {tuner_results['FinalObjectiveValue'].max():.4f}")

# Show all results sorted by performance
print("\nAll tuning jobs (best to worst):")
print(tuner_results.sort_values('FinalObjectiveValue')[
    ['TrainingJobName', 'FinalObjectiveValue', 'max_depth', 'eta', 'min_child_weight', 'subsample']
].to_string(index=False))

print("\n📸 SCREENSHOT: Open SageMaker Console > Hyperparameter tuning jobs and take a screenshot.")


## Phase 1D: Deploy & Test the Model

Now we'll deploy the **best model** from tuning to a real-time endpoint. This creates a persistent HTTPS endpoint that you can send data to and get predictions back in real time.


In [ ]:
# Deploy the best model from tuning
# DO NOT change instance_type — ml.m5.large is the smallest supported for endpoints
print("Deploying endpoint... (this will take ~5 minutes)")
predictor = tuner.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',  # DO NOT CHANGE — Learner Lab restriction
    serializer=sagemaker.serializers.CSVSerializer()
)
print(f"Endpoint deployed: {predictor.endpoint_name}")


In [ ]:
# Test predictions on the held-out test set
from sagemaker.deserializers import CSVDeserializer

predictor.deserializer = CSVDeserializer()

# Predict on test set (features only, no target column)
predictions_raw = predictor.predict(X_test[:20])
predictions = np.array([float(p) for row in predictions_raw for p in row])

# Compare with actual values
print("Predictions vs Actual (first 20 test samples):")
print(f"{'Predicted':>12} {'Actual':>12} {'Error':>12}")
print("-" * 38)
for pred, actual in zip(predictions[:20], y_test[:20]):
    print(f"{pred:12.2f} {actual:12.2f} {abs(pred-actual):12.2f}")

# Calculate RMSE
rmse = np.sqrt(np.mean((predictions - y_test[:20]) ** 2))
print(f"\nRMSE on test sample: {rmse:.4f}")


In [ ]:
# TODO: Test with custom examples
# =============================================
# Create 3 test cases for restaurants that DON'T exist in the dataset.
# Use the same feature encoding as your training data.
#
# Think about:
# - What would the model predict for a new $$$ Italian restaurant in South End?
# - What about a $ Mexican place in University City?
# - How about a $$$$ steakhouse in Uptown?
#
# You'll need to one-hot encode your test cases the same way as the training data.
# Remember the column order must match exactly.
#
# Example structure (adjust columns to match YOUR feature encoding):
# test_case = np.array([[price_range, review_count, outdoor, delivery, reservations,
#                         avg_review_len, pct_positive, ...cuisine_dummies..., ...neighborhood_dummies...]])

# Test case 1: Describe what restaurant this represents
# test_1 = ...
# pred_1 = predictor.predict(test_1)
# print(f"Test 1 — [your description]: Predicted rating = {pred_1}")

# Test case 2:
# ...

# Test case 3:
# ...
# =============================================


### Feature Importance Interpretation

**TODO:** Write 3-4 sentences interpreting what you've observed:
- Which features seem to matter most for predicting restaurant ratings?
- Do the important features make intuitive sense?
- Were there any surprises in how the model uses the features?

*Write your interpretation in the cell below:*


**Your interpretation:**

*(TODO: Write 3-4 sentences here)*



---

**IMPORTANT:** Do NOT delete the endpoint yet! You will delete it in Part 3 (Cleanup) after completing Part 2.

Proceed to **Part 2: GenAI with Flan-T5** notebook.
